# Hindsight GST Invoice CBA — Tax Component Concept Binding**Goal**: Measure concept-binding accuracy on Indian GST (Goods and Services Tax) invoices, targeting the highly confusable tax component fields (CGST, SGST, IGST).**Dataset**: [Roboflow Tax Invoice](https://universe.roboflow.com/writer-information-6sirk/tax-invoice-ud2jo) — 773 Indian GST invoice images with 24 field categories in COCO format (CC BY 4.0).**Why GST invoices?** Indian GST invoices are an ideal concept-binding challenge:- **Tax trio**: CGST, SGST, and IGST often have identical or near-identical values on the same invoice- **Amount confusion**: Initial Total Amount (pre-tax) vs Total Amount (post-tax) vs Round Off- **Date pair**: Invoice Date vs Acknowledgement Date- **Identifier pair**: Invoice Number vs Ack No. vs GSTIN vs CIN No.**Methodology**:- **GT construction**: Claude Sonnet 4.6 with *guided* extraction (told which fields are present via COCO annotations)- **Test models**: Claude Haiku 4.5 and GPT-4o-mini with *unguided* extraction (discover and map fields independently)- **Vision-only**: Images are 640×640 (resized from originals); Tesseract OCR is unreliable at this resolution- **Scoring**: Value-first matching with CBA-strict and CBA-soft metrics

In [ ]:
# Cell 1: Setup & Dependencies

!pip install anthropic openai Pillow roboflow -q

import anthropic, openai, json, time, os, re, random, base64, io
from PIL import Image
from collections import defaultdict, Counter
from datetime import datetime
from google.colab import userdata


ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# Download Roboflow Tax Invoice dataset (COCO format)
DATASET_DIR = "Tax-invoice-1"
if not os.path.isdir(DATASET_DIR):
    from roboflow import Roboflow
    rf = Roboflow(api_key="ZJuS6aqkhbJLXlPDjlQK")
    project = rf.workspace("writer-information-6sirk").project("tax-invoice-ud2jo")
    dataset = project.version(1).download("coco")
    DATASET_DIR = dataset.location
    print(f"Downloaded to: {DATASET_DIR}")
else:
    print(f"Dataset already present at {DATASET_DIR}")

assert os.path.isdir(DATASET_DIR), f"Dataset not found at {DATASET_DIR}"
print("Setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 63.9 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Tax-invoice-1 in coco:: 100%|██████████| 781/781 [00:00<00:00, 4373.73it/s]


Downloaded to: /content/Tax-invoice-1
Setup complete.


In [ ]:
# Cell 2: Load COCO Annotations from All Splits

def load_coco_split(split_dir):
    """Load COCO annotations for one split, return list of doc dicts."""
    ann_path = os.path.join(DATASET_DIR, split_dir, "_annotations.coco.json")
    with open(ann_path) as f:
        data = json.load(f)

    cat_map = {c["id"]: c["name"] for c in data["categories"]}

    # Build per-image annotation lists
    img_map = {img["id"]: img for img in data["images"]}
    img_anns = defaultdict(list)
    for ann in data["annotations"]:
        cat_name = cat_map[ann["category_id"]]
        img_anns[ann["image_id"]].append({
            "category": cat_name,
            "bbox": ann["bbox"],
        })

    docs = []
    for img_id, img_info in img_map.items():
        docs.append({
            "image_path": os.path.join(DATASET_DIR, split_dir, img_info["file_name"]),
            "file_name": img_info["file_name"],
            "width": img_info["width"],
            "height": img_info["height"],
            "annotations": img_anns.get(img_id, []),
            "split": split_dir,
        })
    return docs

all_docs = []
for split in ["train", "valid", "test"]:
    split_docs = load_coco_split(split)
    print(f"  {split}: {len(split_docs)} images")
    all_docs.extend(split_docs)

# Verify image files exist
found = sum(1 for d in all_docs if os.path.isfile(d["image_path"]))
print(f"\nTotal: {len(all_docs)} images ({found} files found)")

  train: 542 images
  valid: 154 images
  test: 77 images

Total: 773 images (773 files found)


In [ ]:
# Cell 3: Eval Concepts, Ontology & Prompts

# ── COCO category → canonical eval concept mapping ──
COCO_TO_CONCEPT = {
    "CGST": "cgst",
    "SGST": "sgst",
    "IGST": "igst",
    "Initial Total Amount": "initial_total_amount",
    "Total Amount": "total_amount",
    "Round off": "round_off",
    "Invoice Number": "invoice_number",
    "Invoice Date": "invoice_date",
    "Ack No.": "ack_no",
    "Ack Date": "ack_date",
    "GST Number": "gst_number",
    "CIN No.": "cin_no",
    "Amount in Words": "amount_in_words",
}

EVAL_CONCEPTS = sorted(set(COCO_TO_CONCEPT.values()))

# ── Ontology: 4 families ──
GST_ONTOLOGY = {
    "families": {
        "tax_components": ["cgst", "sgst", "igst"],
        "amounts": ["initial_total_amount", "total_amount", "round_off", "amount_in_words"],
        "dates": ["invoice_date", "ack_date"],
        "identifiers": ["invoice_number", "ack_no", "gst_number", "cin_no"],
    }
}

concept_to_family = {}
for family, concepts in GST_ONTOLOGY["families"].items():
    for c in concepts:
        concept_to_family[c] = family
GST_ONTOLOGY["concept_to_family"] = concept_to_family

# ── Map COCO annotations to eval concepts per document ──
for doc in all_docs:
    present = set()
    for ann in doc["annotations"]:
        concept = COCO_TO_CONCEPT.get(ann["category"])
        if concept:
            present.add(concept)
    doc["eval_concepts"] = sorted(present)

# Filter: >= 3 eval concepts present
qualified = [d for d in all_docs if len(d["eval_concepts"]) >= 3]
print(f"Images with >= 3 eval concepts: {len(qualified)} / {len(all_docs)}")

# Concept coverage
concept_counts = Counter()
for d in qualified:
    for c in d["eval_concepts"]:
        concept_counts[c] += 1
print("\nConcept coverage in qualified images:")
for c, n in concept_counts.most_common():
    print(f"  {c:25s} [{concept_to_family.get(c,'?'):15s}] {n:4d}")

# ── GT prompt (guided — Sonnet knows which fields to extract) ──
GT_SYSTEM = """You are a document extraction system specialized in Indian GST tax invoices.
You will be told which specific fields are present in the invoice. Extract the exact text value for each.
Return a JSON object mapping each field name to its extracted value.
Only include fields from the provided list. Do not fabricate values — only extract text visible on the invoice."""

# ── Test prompt (unguided — test models discover fields) ──
TEST_SYSTEM = """You are a document extraction system. Look at this Indian GST tax invoice and extract fields, mapping each to the most appropriate canonical concept.

Canonical concepts:
cgst, sgst, igst, initial_total_amount, total_amount, round_off, invoice_number, invoice_date, ack_no, ack_date, gst_number, cin_no, amount_in_words

Return a JSON array of objects. Each object must have:
- "label": the field label as shown on the invoice
- "value": the corresponding value as shown on the invoice
- "concept": the canonical concept name from the list above

Rules:
- Only include fields that clearly match a canonical concept
- Each concept may appear at most once
- Do not fabricate values — only use text visible on the invoice
- CGST = Central GST, SGST = State GST, IGST = Integrated GST
- initial_total_amount = taxable amount before tax, total_amount = final total after tax
- gst_number = GSTIN (15-character alphanumeric)"""

print(f"\nEval concepts ({len(EVAL_CONCEPTS)}): {EVAL_CONCEPTS}")

Images with >= 3 eval concepts: 663 / 773

Concept coverage in qualified images:
  invoice_number            [identifiers    ]  630
  invoice_date              [dates          ]  621
  total_amount              [amounts        ]  596
  amount_in_words           [amounts        ]  410
  initial_total_amount      [amounts        ]  278
  gst_number                [identifiers    ]  226
  sgst                      [tax_components ]  190
  cgst                      [tax_components ]  189
  igst                      [tax_components ]  174
  round_off                 [amounts        ]  113
  ack_no                    [identifiers    ]   80
  cin_no                    [identifiers    ]   76
  ack_date                  [dates          ]   72

Eval concepts (13): ['ack_date', 'ack_no', 'amount_in_words', 'cgst', 'cin_no', 'gst_number', 'igst', 'initial_total_amount', 'invoice_date', 'invoice_number', 'round_off', 'sgst', 'total_amount']


In [ ]:
# Cell 4: Helper Functions

MAX_RETRIES = 5
BASE_DELAY_ANTHROPIC = 1.5
BASE_DELAY_OPENAI = 5
MAX_IMAGE_BYTES = 4_500_000


def _parse_json(raw):
    """Strip markdown fences and parse JSON."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)


def _retry_with_backoff(fn, max_retries=MAX_RETRIES, base_delay=2):
    """Call fn() with exponential backoff. Rate-limit-aware: 10s, 20s, 40s, 80s..."""
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            err_str = str(e).lower()
            is_rate_limit = "429" in str(e) or "rate_limit" in err_str or "rate limit" in err_str
            if attempt < max_retries - 1:
                if is_rate_limit:
                    delay = min(10 * (2 ** attempt), 120)
                else:
                    delay = base_delay * (2 ** attempt)
                label = "Rate limited" if is_rate_limit else "Error"
                print(f"    {label}, retry {attempt+1}/{max_retries-1} in {delay:.0f}s: {type(e).__name__}")
                time.sleep(delay)
            else:
                raise


def _image_to_base64(image_path):
    """Read image, resize if needed to stay under API limit, return (base64_str, media_type)."""
    file_size = os.path.getsize(image_path)
    if file_size <= MAX_IMAGE_BYTES:
        with open(image_path, "rb") as f:
            data = f.read()
        ext = image_path.lower()
        media_type = "image/jpeg" if ext.endswith((".jpg", ".jpeg")) else "image/png"
        return base64.b64encode(data).decode("utf-8"), media_type
    # Resize if too large
    img = Image.open(image_path)
    for scale in [0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2]:
        new_w, new_h = int(img.width * scale), int(img.height * scale)
        resized = img.resize((new_w, new_h), Image.LANCZOS)
        if resized.mode == "RGBA":
            resized = resized.convert("RGB")
        buf = io.BytesIO()
        resized.save(buf, format="JPEG", quality=85)
        if buf.tell() <= MAX_IMAGE_BYTES:
            return base64.b64encode(buf.getvalue()).decode("utf-8"), "image/jpeg"
    raise ValueError(f"Cannot shrink {image_path} under {MAX_IMAGE_BYTES} bytes")


def _array_to_predictions(stage2_array):
    """Convert [{label, value, concept}, ...] array to {concept: value} dict."""
    predictions = {}
    for item in stage2_array:
        if not isinstance(item, dict):
            continue
        concept = item.get("concept", "")
        value = item.get("value", "")
        if concept and concept not in predictions:
            predictions[concept] = value
    return predictions


print("Helper functions ready.")

Helper functions ready.


In [ ]:
# Cell 5: GT Construction — Sonnet 4.6 Guided Extraction
# Sonnet is told WHICH fields to look for (from COCO annotations).
# This is a different task than unguided test extraction.

GT_MODEL = "claude-sonnet-4-6"
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Sample candidates: up to 150, then keep first 100 with valid GT
random.seed(42)
if len(qualified) > 150:
    candidates = random.sample(qualified, 150)
else:
    candidates = qualified[:]
random.shuffle(candidates)
print(f"GT candidates: {len(candidates)}")


def extract_gt_sonnet(doc):
    """Use Sonnet with guided extraction to build GT for one document."""
    concept_list = ", ".join(doc["eval_concepts"])
    user_msg = f"Extract the value for each of these fields present in this invoice: {concept_list}"

    img_b64, media_type = _image_to_base64(doc["image_path"])

    def _call():
        resp = ant_client.messages.create(
            model=GT_MODEL,
            max_tokens=1024,
            system=GT_SYSTEM,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": img_b64}},
                    {"type": "text", "text": user_msg},
                ],
            }],
        )
        raw = resp.content[0].text
        return _parse_json(raw)

    return _retry_with_backoff(_call, base_delay=BASE_DELAY_ANTHROPIC)


# Run GT extraction
gt_docs = []
gt_errors = 0
for i, doc in enumerate(candidates):
    try:
        gt_values = extract_gt_sonnet(doc)
        # Filter to eval concepts with non-empty string values
        gt = {}
        for concept in doc["eval_concepts"]:
            val = gt_values.get(concept, "")
            if isinstance(val, (int, float)):
                val = str(val)
            if isinstance(val, str) and val.strip():
                gt[concept] = val.strip()

        if len(gt) >= 3:
            doc["gt"] = gt
            gt_docs.append(doc)
            if len(gt_docs) % 10 == 0:
                print(f"  GT built: {len(gt_docs)} docs ({i+1}/{len(candidates)} processed)")

        if len(gt_docs) >= 100:
            print(f"  Reached 100 docs, stopping GT construction.")
            break

        time.sleep(BASE_DELAY_ANTHROPIC)
    except Exception as e:
        gt_errors += 1
        print(f"  GT error on {doc['file_name']}: {e}")

samples = gt_docs
print(f"\nGT construction complete: {len(samples)} documents with valid GT ({gt_errors} errors)")

# GT stats
gt_concept_counts = Counter()
for d in samples:
    for c in d["gt"]:
        gt_concept_counts[c] += 1
print("\nGT concept distribution:")
for c, n in gt_concept_counts.most_common():
    family = concept_to_family.get(c, "?")
    print(f"  {c:25s} [{family:15s}] {n:4d}")

# Count docs with duplicate values (prime misbinding targets)
dup_val_docs = 0
for d in samples:
    vals = list(d["gt"].values())
    if len(vals) != len(set(vals)):
        dup_val_docs += 1
print(f"\nDocs with duplicate values across concepts: {dup_val_docs}/{len(samples)}")
print("  (These are prime targets for misbinding — e.g., CGST=SGST)")

GT candidates: 150
    Error, retry 1/4 in 2s: JSONDecodeError
    Error, retry 2/4 in 3s: JSONDecodeError
  GT built: 10 docs (10/150 processed)
    Error, retry 1/4 in 2s: JSONDecodeError
    Error, retry 2/4 in 3s: JSONDecodeError
    Error, retry 3/4 in 6s: JSONDecodeError
    Error, retry 4/4 in 12s: JSONDecodeError
  GT error on 4519040374_S669122-23_jpg.rf.88dbcfcc7a388cc1bfb2113c30b5d6ef.jpg: Expecting value: line 1 column 1 (char 0)
  GT built: 20 docs (21/150 processed)
  GT built: 30 docs (31/150 processed)
    Error, retry 1/4 in 2s: JSONDecodeError
    Error, retry 2/4 in 3s: JSONDecodeError
    Error, retry 3/4 in 6s: JSONDecodeError
  GT built: 40 docs (41/150 processed)
  GT built: 50 docs (51/150 processed)
    Error, retry 1/4 in 2s: JSONDecodeError
  GT built: 60 docs (61/150 processed)
    Error, retry 1/4 in 2s: JSONDecodeError
    Error, retry 2/4 in 3s: JSONDecodeError
    Error, retry 3/4 in 6s: JSONDecodeError
    Error, retry 4/4 in 12s: JSONDecodeError
  GT e

In [ ]:
# Cell 6: Scoring Functions — Value-First Matching

def normalize_value(v):
    """Normalize a value for comparison. Handles Indian currency formats."""
    v = str(v).lower().strip()
    # Strip currency symbols
    for sym in ["₹", "rs.", "rs", "inr", "/-"]:
        v = v.replace(sym, "")
    v = v.replace(",", "").strip()
    v = re.sub(r"\s+", " ", v)
    # Strip trailing zeros on decimals: "49.50" → "49.5", "100.00" → "100"
    if re.fullmatch(r"-?\d+\.\d+", v):
        v = v.rstrip("0").rstrip(".")
    # Normalize dash/nil to empty
    if v in ["-", "nil", "n/a", "na", "—", "–"]:
        v = "0"
    return v


def cba_strict(predicted_concept, ground_truth_concept):
    return 1.0 if predicted_concept == ground_truth_concept else 0.0


def cba_soft(predicted_concept, ground_truth_concept, ontology):
    if predicted_concept == ground_truth_concept:
        return 1.0
    c2f = ontology.get("concept_to_family", {})
    if c2f.get(ground_truth_concept) and c2f.get(ground_truth_concept) == c2f.get(predicted_concept):
        return 0.5
    return 0.0


def score_field(gt_value, pred_value, gt_concept, pred_concept, ontology):
    norm_gt = normalize_value(gt_value)
    norm_pred = normalize_value(pred_value)
    value_match = norm_gt == norm_pred
    strict = cba_strict(pred_concept, gt_concept)
    soft = cba_soft(pred_concept, gt_concept, ontology)
    return {
        "value_match": value_match,
        "field_recall_contribution": 1.0 if value_match else 0.0,
        "cba_strict": strict,
        "cba_soft": soft,
        "is_misbinding": value_match and strict == 0.0,
        "gt_concept": gt_concept,
        "pred_concept": pred_concept,
        "gt_value": gt_value,
        "pred_value": pred_value,
    }


def score_document(ground_truth, predictions, ontology, eval_concepts=None):
    """Score all fields for a document using VALUE-FIRST matching.

    1. For each GT concept, search ALL predictions for a matching value
    2. Prefer exact concept match when value appears under multiple keys
    3. Value found under wrong concept → misbinding
    4. No value match → fall back to concept-key match
    5. Neither → complete miss
    """
    if eval_concepts is not None:
        ground_truth = {k: v for k, v in ground_truth.items() if k in eval_concepts}

    per_field = []
    misbindings = []
    used_pred_keys = set()

    for gt_concept, gt_value in ground_truth.items():
        norm_gt = normalize_value(gt_value)

        # Step 1: Value-first — find this value anywhere in predictions
        value_matches = []
        for pred_concept, pred_value in predictions.items():
            if pred_concept in used_pred_keys:
                continue
            norm_pred = normalize_value(pred_value)
            if norm_gt == norm_pred:
                value_matches.append(pred_concept)

        if value_matches:
            # Prefer exact concept match
            if gt_concept in value_matches:
                chosen = gt_concept
            else:
                chosen = value_matches[0]
            used_pred_keys.add(chosen)
            result = score_field(gt_value, predictions[chosen], gt_concept, chosen, ontology)
            per_field.append(result)
            if result["is_misbinding"]:
                misbindings.append(result)
            continue

        # Step 2: Concept-key match (value mismatch)
        if gt_concept in predictions and gt_concept not in used_pred_keys:
            used_pred_keys.add(gt_concept)
            result = score_field(gt_value, predictions[gt_concept], gt_concept, gt_concept, ontology)
            per_field.append(result)
            continue

        # Step 3: Complete miss
        per_field.append({
            "value_match": False,
            "field_recall_contribution": 0.0,
            "cba_strict": 0.0,
            "cba_soft": 0.0,
            "is_misbinding": False,
            "gt_concept": gt_concept,
            "pred_concept": None,
            "gt_value": gt_value,
            "pred_value": None,
        })

    # Aggregate
    n = len(per_field)
    field_recall = sum(f["field_recall_contribution"] for f in per_field) / n if n else 0.0
    cba_s = sum(f["cba_strict"] for f in per_field) / n if n else 0.0
    cba_soft_avg = sum(f["cba_soft"] for f in per_field) / n if n else 0.0

    return {
        "field_recall": field_recall,
        "cba_strict": cba_s,
        "cba_soft": cba_soft_avg,
        "delta": field_recall - cba_s,
        "n_gt_fields": n,
        "n_pred_fields": len(predictions),
        "misbindings": misbindings,
        "per_field": per_field,
    }


# Sanity check: known misbinding
_test_gt = {"cgst": "49.50", "sgst": "49.50", "total_amount": "649"}
_test_pred = {"cgst": "649", "sgst": "49.50", "total_amount": "49.50"}
_test_score = score_document(_test_gt, _test_pred, GST_ONTOLOGY)
assert _test_score["delta"] > 0, "Sanity check failed: should detect misbinding"
assert len(_test_score["misbindings"]) > 0, "Sanity check failed: should have misbinding"
print(f"Sanity check passed: delta={_test_score['delta']:.3f}, misbindings={len(_test_score['misbindings'])}")
print("Scoring functions ready.")

Sanity check passed: delta=1.000, misbindings=3
Scoring functions ready.


In [ ]:
# Cell 7: Extraction Functions — Vision-Based (Haiku + GPT-4o-mini)

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY)


def vision_call_anthropic(image_path, system_prompt, model_id):
    """Call Anthropic vision API and return parsed JSON array."""
    img_b64, media_type = _image_to_base64(image_path)

    def _call():
        resp = ant_client.messages.create(
            model=model_id,
            max_tokens=1024,
            system=system_prompt,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": img_b64}},
                    {"type": "text", "text": "Extract all matching fields from this GST invoice."},
                ],
            }],
        )
        raw = resp.content[0].text
        return _parse_json(raw)

    return _retry_with_backoff(_call, base_delay=BASE_DELAY_ANTHROPIC)


def vision_call_openai(image_path, system_prompt, model_id):
    """Call OpenAI vision API and return parsed JSON array."""
    img_b64, media_type = _image_to_base64(image_path)

    def _call():
        resp = oai_client.chat.completions.create(
            model=model_id,
            max_tokens=1024,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:{media_type};base64,{img_b64}"}},
                    {"type": "text", "text": "Extract all matching fields from this GST invoice."},
                ]},
            ],
        )
        return _parse_json(resp.choices[0].message.content)

    return _retry_with_backoff(_call, base_delay=BASE_DELAY_OPENAI)


def extract_vision(image_path, model_id, provider):
    """Run vision extraction and return (stage2_array, predictions)."""
    if provider == "anthropic":
        stage2_array = vision_call_anthropic(image_path, TEST_SYSTEM, model_id)
    else:
        stage2_array = vision_call_openai(image_path, TEST_SYSTEM, model_id)
    predictions = _array_to_predictions(stage2_array)
    return stage2_array, predictions


print("Extraction functions ready.")

Extraction functions ready.


In [ ]:
# Cell 8: Run Loop — 2 models × vision track

MODELS = {
    "haiku": {"provider": "anthropic", "model_id": "claude-haiku-4-5-20251001"},
    "gpt4o-mini": {"provider": "openai", "model_id": "gpt-4o-mini"},
}

RATE_DELAYS = {"anthropic": 1.5, "openai": 5}

if not samples:
    raise RuntimeError("No samples with valid GT — check Cell 5 output")

experiments = {}

for model_name, model_cfg in MODELS.items():
    config_key = f"{model_name}_vision"
    provider = model_cfg["provider"]
    model_id = model_cfg["model_id"]
    rate_delay = RATE_DELAYS[provider]

    print(f"\n{'='*60}")
    print(f"Running: {config_key} — {model_id}")
    print(f"{'='*60}")

    doc_results = []
    all_misbindings = []
    errors = []
    raw_outputs = []

    for i, d in enumerate(samples):
        doc_id = f"gst_{i:04d}"
        image_path = d["image_path"]

        try:
            stage2_array, predictions = extract_vision(image_path, model_id, provider)

            raw_outputs.append({
                "doc_id": doc_id,
                "n_mappings": len(stage2_array) if isinstance(stage2_array, list) else 0,
                "mappings": stage2_array,
            })

            gt = d["gt"]
            doc_score = score_document(gt, predictions, GST_ONTOLOGY, eval_concepts=EVAL_CONCEPTS)
            doc_score["doc_id"] = doc_id
            doc_score["file_name"] = d["file_name"]
            doc_score["predictions"] = predictions
            doc_score["gt"] = gt
            doc_score["stage2_array"] = stage2_array
            doc_results.append(doc_score)

            for mb in doc_score["misbindings"]:
                mb["doc_id"] = doc_id
                mb["model"] = model_name
                all_misbindings.append(mb)

        except Exception as e:
            print(f"  ERROR on {doc_id}: {e}")
            errors.append({"doc_id": doc_id, "error": str(e)})

        if (i + 1) % 10 == 0:
            n_done = len(doc_results)
            avg_f1 = sum(r["field_recall"] for r in doc_results) / n_done if n_done else 0
            avg_cba = sum(r["cba_strict"] for r in doc_results) / n_done if n_done else 0
            n_mb = len(all_misbindings)
            print(f"  [{i+1}/{len(samples)}] F1={avg_f1:.3f} CBA={avg_cba:.3f} misbindings={n_mb}")

        time.sleep(rate_delay)

    # Aggregate
    n = len(doc_results)
    if n > 0:
        field_recall = sum(r["field_recall"] for r in doc_results) / n
        cba_strict_avg = sum(r["cba_strict"] for r in doc_results) / n
        cba_soft_avg = sum(r["cba_soft"] for r in doc_results) / n
        delta = field_recall - cba_strict_avg
    else:
        field_recall = cba_strict_avg = cba_soft_avg = delta = 0.0

    experiments[config_key] = {
        "model_name": model_name,
        "model_id": model_id,
        "track_name": "LLM Vision",
        "field_recall": field_recall,
        "cba_strict": cba_strict_avg,
        "cba_soft": cba_soft_avg,
        "delta": delta,
        "total_misbindings": len(all_misbindings),
        "misbindings": all_misbindings,
        "per_document": doc_results,
        "raw_outputs": raw_outputs,
        "errors": errors,
        "n_docs": n,
    }

    print(f"\n  RESULTS: F1={field_recall:.4f} CBA-strict={cba_strict_avg:.4f} CBA-soft={cba_soft_avg:.4f} Delta={delta:.4f}")
    print(f"  Misbindings: {len(all_misbindings)} | Errors: {len(errors)}")

print("\n" + "="*60)
print("All experiments complete.")


Running: haiku_vision — claude-haiku-4-5-20251001
  [10/100] F1=0.227 CBA=0.725 misbindings=1
  [20/100] F1=0.282 CBA=0.742 misbindings=2
  [30/100] F1=0.290 CBA=0.742 misbindings=2
  [40/100] F1=0.314 CBA=0.767 misbindings=2
  [50/100] F1=0.284 CBA=0.742 misbindings=3
  [60/100] F1=0.284 CBA=0.730 misbindings=3
  [70/100] F1=0.312 CBA=0.742 misbindings=3
  [80/100] F1=0.318 CBA=0.736 misbindings=4
  [90/100] F1=0.313 CBA=0.745 misbindings=6
  [100/100] F1=0.316 CBA=0.758 misbindings=6

  RESULTS: F1=0.3158 CBA-strict=0.7584 CBA-soft=0.7623 Delta=-0.4426
  Misbindings: 6 | Errors: 0

Running: gpt4o-mini_vision — gpt-4o-mini
  [10/100] F1=0.192 CBA=0.692 misbindings=1
  [20/100] F1=0.348 CBA=0.822 misbindings=1
  [30/100] F1=0.340 CBA=0.786 misbindings=1
    Error, retry 1/4 in 5s: JSONDecodeError
  [40/100] F1=0.331 CBA=0.795 misbindings=1
    Error, retry 1/4 in 5s: JSONDecodeError
  [50/100] F1=0.284 CBA=0.757 misbindings=2
  [60/100] F1=0.285 CBA=0.752 misbindings=2
  [70/100] F1=0

In [ ]:
# Cell 9: Results Dashboard

print("=" * 85)
print("GST INVOICE CBA — RESULTS SUMMARY")
print("=" * 85)

# Cross-model comparison
print(f"\n{'Model':<15} {'F1':>8} {'CBA-str':>8} {'CBA-soft':>8} {'Delta':>8} {'Misbind':>8} {'Errors':>7}")
print("-" * 72)
for config_key, r in experiments.items():
    print(f"{r['model_name']:<15} {r['field_recall']:>8.4f} {r['cba_strict']:>8.4f} {r['cba_soft']:>8.4f} "
          f"{r['delta']:>8.4f} {r['total_misbindings']:>8d} {len(r['errors']):>7d}")

# Per-concept breakdown
print(f"\n{'='*85}")
print("PER-CONCEPT ACCURACY")
print("=" * 85)

for config_key, results in experiments.items():
    print(f"\n--- {config_key} ---")
    concept_stats = defaultdict(lambda: {"field_acc": [], "cba_strict": [], "cba_soft": [], "misbindings": 0})

    for doc in results["per_document"]:
        for field in doc["per_field"]:
            gt_c = field["gt_concept"]
            if gt_c in EVAL_CONCEPTS:
                concept_stats[gt_c]["field_acc"].append(field["field_recall_contribution"])
                concept_stats[gt_c]["cba_strict"].append(field["cba_strict"])
                concept_stats[gt_c]["cba_soft"].append(field["cba_soft"])
                if field["is_misbinding"]:
                    concept_stats[gt_c]["misbindings"] += 1

    print(f"  {'Concept':<25} {'Family':<16} {'N':>4} {'F1':>6} {'CBA':>6} {'Delta':>7} {'Misbind':>8}")
    print(f"  {'-'*72}")
    for concept in sorted(EVAL_CONCEPTS):
        stats = concept_stats[concept]
        n = len(stats["field_acc"])
        if n > 0:
            fa = sum(stats["field_acc"]) / n
            cs = sum(stats["cba_strict"]) / n
            delta = fa - cs
            fam = concept_to_family.get(concept, "?")
            print(f"  {concept:<25} {fam:<16} {n:>4} {fa:>6.3f} {cs:>6.3f} {delta:>+7.3f} {stats['misbindings']:>8}")

# Misbinding details
print(f"\n{'='*85}")
print("MISBINDING DETAILS")
print("=" * 85)
for config_key, results in experiments.items():
    if results["misbindings"]:
        print(f"\n--- {config_key}: {len(results['misbindings'])} misbindings ---")
        for mb in results["misbindings"][:20]:
            print(f"  {mb['doc_id']}: GT={mb['gt_concept']} → Pred={mb['pred_concept']} "
                  f"value=\"{mb['gt_value']}\"")
    else:
        print(f"\n--- {config_key}: 0 misbindings ---")

GST INVOICE CBA — RESULTS SUMMARY

Model                 F1  CBA-str CBA-soft    Delta  Misbind  Errors
------------------------------------------------------------------------
haiku             0.3158   0.7584   0.7623  -0.4426        6       0
gpt4o-mini        0.3300   0.7977   0.8007  -0.4677        5       0

PER-CONCEPT ACCURACY

--- haiku_vision ---
  Concept                   Family              N     F1    CBA   Delta  Misbind
  ------------------------------------------------------------------------
  ack_date                  dates               8  0.375  0.875  -0.500        0
  ack_no                    identifiers         9  0.000  0.889  -0.889        0
  amount_in_words           amounts            59  0.153  0.424  -0.271        0
  cgst                      tax_components     27  0.593  0.889  -0.296        0
  cin_no                    identifiers        16  0.000  0.375  -0.375        0
  gst_number                identifiers        39  0.051  0.744  -0.692        0

In [ ]:
# Cell 10: Confusion Matrix & Export

# Build confusion matrix (predicted vs GT concept)
print("CONCEPT CONFUSION MATRIX")
print("=" * 85)

for config_key, results in experiments.items():
    print(f"\n--- {config_key} ---")
    confusion = defaultdict(Counter)
    for doc in results["per_document"]:
        for field in doc["per_field"]:
            gt_c = field["gt_concept"]
            pred_c = field.get("pred_concept") or "MISS"
            confusion[gt_c][pred_c] += 1

    # Show off-diagonal entries (confusions)
    print("  Off-diagonal entries (confusions):")
    has_confusion = False
    for gt_c in sorted(EVAL_CONCEPTS):
        for pred_c, count in sorted(confusion[gt_c].items()):
            if pred_c != gt_c and pred_c != "MISS" and count > 0:
                fam_gt = concept_to_family.get(gt_c, "?")
                fam_pred = concept_to_family.get(pred_c, "?")
                same_fam = "SAME" if fam_gt == fam_pred else "DIFF"
                print(f"    {gt_c:25s} → {pred_c:25s} ×{count} [{same_fam} family]")
                has_confusion = True
    if not has_confusion:
        print("    (no concept confusions detected)")

    # Show miss counts
    print("  Miss counts:")
    for gt_c in sorted(EVAL_CONCEPTS):
        miss_count = confusion[gt_c].get("MISS", 0)
        total = sum(confusion[gt_c].values())
        if total > 0:
            print(f"    {gt_c:25s} {miss_count}/{total} missed ({miss_count/total*100:.0f}%)")

# ── Export Results ──
output = {
    "experiment": "hindsight_gst_invoice_cba",
    "dataset": "Roboflow Tax Invoice v1",
    "dataset_url": "https://universe.roboflow.com/writer-information-6sirk/tax-invoice-ud2jo",
    "license": "CC BY 4.0",
    "n_images": len(samples),
    "gt_model": GT_MODEL,
    "eval_concepts": EVAL_CONCEPTS,
    "ontology": GST_ONTOLOGY["families"],
    "prompt_design": "unguided vision extraction with array output",
    "scoring": "value-first matching, CBA-strict + CBA-soft",
    "timestamp": datetime.now().isoformat(),
    "results": {},
}

for config_key, results in experiments.items():
    output["results"][config_key] = {
        "model_name": results["model_name"],
        "model_id": results["model_id"],
        "field_recall": results["field_recall"],
        "cba_strict": results["cba_strict"],
        "cba_soft": results["cba_soft"],
        "delta": results["delta"],
        "total_misbindings": results["total_misbindings"],
        "n_docs": results["n_docs"],
        "n_errors": len(results["errors"]),
        "misbinding_details": [
            {"doc_id": mb["doc_id"], "gt_concept": mb["gt_concept"],
             "pred_concept": mb["pred_concept"], "value": mb["gt_value"]}
            for mb in results["misbindings"]
        ],
        "per_document": [
            {"doc_id": d["doc_id"], "file_name": d["file_name"],
             "field_recall": d["field_recall"], "cba_strict": d["cba_strict"],
             "delta": d["delta"], "n_misbindings": len(d["misbindings"]),
             "gt": d["gt"], "predictions": d["predictions"]}
            for d in results["per_document"]
        ],
    }

results_path = "hindsight_gst_invoice_cba_results.json"
with open(results_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"\nResults exported to {results_path}")
print(f"Total: {sum(r['total_misbindings'] for r in experiments.values())} misbindings across all models")

CONCEPT CONFUSION MATRIX

--- haiku_vision ---
  Off-diagonal entries (confusions):
    igst                      → cgst                      ×1 [SAME family]
    igst                      → initial_total_amount      ×1 [DIFF family]
    initial_total_amount      → total_amount              ×1 [SAME family]
    invoice_date              → ack_date                  ×1 [SAME family]
    round_off                 → sgst                      ×1 [DIFF family]
    sgst                      → igst                      ×1 [SAME family]
  Miss counts:
    ack_date                  1/8 missed (12%)
    ack_no                    1/9 missed (11%)
    amount_in_words           34/59 missed (58%)
    cgst                      3/27 missed (11%)
    cin_no                    10/16 missed (62%)
    gst_number                10/39 missed (26%)
    igst                      17/32 missed (53%)
    initial_total_amount      15/46 missed (33%)
    invoice_date              9/94 missed (10%)
    invoice_numb